# Week 9, Day 6: Forge: Week 9 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w009/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("1. cos x is about 1 - x^2/2; at x = 0.3:", round(1 - 0.3 ** 2 / 2, 5), "vs", round(np.cos(0.3), 5))
print("2.", np.array([[2 * 1 * 2, 1 ** 2], [1, 1]]))
H = np.array([[2.0, 3.0], [3.0, 10.0]])
print("3.", H.tolist(), "eigenvalues", np.linalg.eigvalsh(H).round(3), "-> minimum")

In [ ]:
def S(x): return 1 / (1 + np.exp(-(x - 4)))
def F(z, hE, wEE=12.0):
    E, I = z
    return np.array([-E + S(wEE * E - 10 * I + hE), (-I + S(10 * E - 2 * I)) / 2])
def jac(z, hE, h=1e-6):
    J = np.zeros((2, 2))
    for j in range(2):
        e = np.zeros(2); e[j] = h
        J[:, j] = (F(z + e, hE) - F(z - e, hE)) / (2 * h)
    return J
def newton(z, hE, steps=50):
    for _ in range(steps):
        z = z - np.linalg.solve(jac(z, hE), F(z, hE))
    return z
print(newton(np.array([0.02, 0.02]), 0.0))

### Exercise: Part A: where does the circuit start to oscillate?

Sweep the drive `hE` over `hs`. At each value, find the fixed point with Newton's method, starting from the previous value's answer (this is called continuation), and store it in `fps`. Store the largest real part of the Jacobian's eigenvalues in `max_re`, and the first drive at which it is positive in `h_hopf`. Then simulate the circuit just above `h_hopf` to see the oscillation.

In [ ]:
hs = np.linspace(-2, 4, 121)
fps, max_re = [], []
z = np.array([0.02, 0.02])
for h in hs:
    z = ...
    fps.append(z)
    max_re.append(...)
fps, max_re = np.array(fps), np.array(max_re)
h_hopf = ...

zt = fps[np.argmax(max_re > 0)] + 0.01; tr = []
for _ in range(4000):
    zt = zt + 0.02 * F(zt, h_hopf + 0.5); tr.append(zt.copy())
tr = np.array(tr)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(hs, fps[:, 0], color="0.6"); ax[0].scatter(hs, fps[:, 0], c=np.where(max_re < 0, "C2", "C3"), s=8)
ax[0].axvline(h_hopf, ls="--", color="0.5"); ax[0].set_xlabel("drive to E"); ax[0].set_ylabel("E at the fixed point")
ax[1].plot(np.arange(4000) * 0.02, tr[:, 0]); ax[1].set_xlabel("time (units of the E time constant)"); ax[1].set_ylabel("E")
plt.tight_layout(); plt.show()
print("Hopf bifurcation near drive", round(h_hopf, 3))

In [ ]:
import numpy as np
def test_track():
    "fps holds a fixed point for every drive in hs, and max_re the largest real part of its Jacobian's eigenvalues"
    assert fps.shape == (len(hs), 2) and max_re.shape == (len(hs),)
    for k in [0, 40, 100]:
        assert np.linalg.norm(F(fps[k], hs[k])) < 1e-9
        assert np.isclose(max_re[k], np.linalg.eigvals(jac(fps[k], hs[k])).real.max())

def test_hopf():
    "h_hopf is the first drive at which the fixed point is unstable, between 1 and 1.5"
    assert 1.0 < h_hopf < 1.5
    assert h_hopf == hs[np.argmax(max_re > 0)]

def test_complex():
    "at the crossing the eigenvalues are a complex pair: a Hopf bifurcation"
    lam = np.linalg.eigvals(jac(fps[np.argmax(max_re > 0)], h_hopf))
    assert np.abs(lam.imag).max() > 0.1

test_track()
test_hopf()
test_complex()
print("All checks passed")

### Exercise: Part B: fit, error bars and coverage

Write `fit_glm(y)`: run 25 Newton steps on the Poisson log-likelihood (gradient $X^\top(\mathbf{y} - \boldsymbol\lambda)$, Hessian $-X^\top\operatorname{diag}(\boldsymbol\lambda)X$) from zero, and return the weights and their standard errors. Then simulate 200 experiments and record the fraction in which the 95% interval for $w_1$ contains the true value, `coverage`.

In [ ]:
theta = np.repeat(np.linspace(0, 2 * np.pi, 24, endpoint=False), 5)
Xd = np.column_stack([np.ones_like(theta), np.cos(theta), np.sin(theta)])
w_true = np.array([1.5, 0.8, -0.4])

def fit_glm(y):
    w = np.zeros(3)
    for _ in range(25):
        lam = np.exp(Xd @ w)
        g = ...
        H = ...
        w = ...
    lam = np.exp(Xd @ w)
    se = ...
    return w, se

rng = np.random.default_rng(0)
hits = 0
for _ in range(200):
    y = rng.poisson(np.exp(Xd @ w_true))
    w, se = fit_glm(y)
    hits += abs(w[1] - w_true[1]) < 1.96 * se[1]
coverage = hits / 200

w, se = fit_glm(rng.poisson(np.exp(Xd @ w_true)))
print("one fit:", w.round(3), "+/-", (1.96 * se).round(3), "  preferred direction", round(np.degrees(np.arctan2(w[2], w[1])) % 360, 1), "deg")
print("coverage of the 95% intervals:", coverage)

In [ ]:
import numpy as np
def test_fit():
    "fit_glm returns the maximum-likelihood weights and standard errors"
    r = np.random.default_rng(1)
    yv = r.poisson(np.exp(Xd @ w_true))
    w, se = fit_glm(yv)
    lam = np.exp(Xd @ w)
    assert np.allclose(Xd.T @ (yv - lam), 0, atol=1e-6)
    assert np.allclose(se, np.sqrt(np.diag(np.linalg.inv(Xd.T @ (Xd * lam[:, None])))))

def test_coverage():
    "the 95% intervals for w1 contain the true value in 88% to 99% of 200 simulated experiments"
    assert 0.88 <= coverage <= 0.99

test_fit()
test_coverage()
print("All checks passed")

### Exercise: Part C: escaping a saddle

On $f(x, y) = \frac{1}{2}(x^2 - y^2) + \frac{1}{4}y^4$, the origin is a saddle and $(0, \pm 1)$ are minima. Run gradient descent with learning rate 0.1 from $(1, \epsilon)$ for each offset in `offsets`, and count the steps until $|y|$ exceeds 0.5. Store the counts in `escape_steps`. Why does each thousandfold smaller offset add the same number of steps?

In [ ]:
offsets = [1e-1, 1e-3, 1e-6, 1e-9]
escape_steps = []
for eps in offsets:
    x, y = 1.0, eps
    k = 0
    while abs(y) <= 0.5:
        gx, gy = ..., ...
        x, y = x - 0.1 * gx, y - 0.1 * gy
        k += 1
    escape_steps.append(k)
print("offsets", offsets, "-> steps to escape", escape_steps)

In [ ]:
import numpy as np
def test_counts():
    "escape_steps has one count per starting offset, and the counts increase as the offset shrinks"
    assert len(escape_steps) == len(offsets)
    assert np.all(np.diff(escape_steps) > 0)

def test_log():
    "each thousandfold smaller offset adds roughly the same number of steps, about ln(1000) / ln(1.1)"
    d = np.diff(escape_steps)
    assert np.all(np.abs(d[1:] - np.log(1000) / np.log(1.1)) < 5)

test_counts()
test_log()
print("All checks passed")